# 10. 비대칭 OD 분석

## 분석 배경 및 목적

도시 내 택시 통행은 본질적으로 비대칭적이다. Liu et al. (2013)은 택시 GPS 데이터를 활용하여 도시의 통행 패턴과 공간 구조를 밝혀냈는데, 특히 출퇴근 시간대에 OD(Origin-Destination) 쌍의 양방향 통행량이 극단적으로 불균형하다는 점을 발견했다. 이러한 비대칭성은 택시 운영에 직접적 비효율을 초래한다: 한 방향으로만 수요가 집중되면 반대 방향은 빈차로 복귀해야 하기 때문이다.

본 분석의 목적:

1. **비대칭 OD 쌍 식별**: 양방향 통행량의 불균형이 큰 행정동 쌍을 찾아낸다
2. **시간대별 OD 반전 패턴**: 출근(7-9시)과 퇴근(17-19시)에 방향이 반전되는 OD 쌍 탐지
3. **빈차 낭비 규모 추정**: 비대칭 OD에서 발생하는 편도 빈차 복귀의 경제적 비용 산출

비대칭 지수(Asymmetry Index)는 `|A→B - B→A| / (A→B + B→A)`로 정의하며, 0이면 완전 대칭, 1이면 완전 편도 수요를 의미한다.


In [ ]:
import gc, psutil, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# plt.rcParams['font.family'] = 'AppleGothic'  # Mac
plt.rcParams['font.family'] = 'Malgun Gothic'   # Windows
plt.rcParams['axes.unicode_minus'] = False

CHUNK_SIZE = 1_000_000
D012_PATH = './DC_TBYXD012.csv'

DTYPE_D012 = {
    'PAY_AMT': 'int32',
    'RIDE_DIST': 'float32',
    'VACNTV_DIST': 'float32',
    'RIDE_A_CD': 'category',
    'ALIGHT_A_CD': 'category',
    'RIDE_POS_X': 'float32',
    'RIDE_POS_Y': 'float32',
    'ALIGHT_POS_X': 'float32',
    'ALIGHT_POS_Y': 'float32',
    'DRIVER_ID': 'category',
    'TAXI_VEHC_ID': 'category',
    'PLTF_FEE_AMT': 'int32',
}

def mem_usage():
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

mem_usage()

## 1. 행정동 쌍별 양방향 통행량 산출 (chunk별 집계)

OD 비대칭성을 분석하려면 먼저 모든 행정동 쌍에 대해 양방향 통행량을 집계해야 한다. Liu et al. (2013)의 접근을 따라, 승차 행정동(Origin)과 하차 행정동(Destination)을 기준으로 쌍별 통행량을 산출한다. 대용량 데이터(약 6억 건)를 처리하기 위해 chunk 단위로 집계 후 합산한다.


In [ ]:
# OD쌍별 통행량 집계
od_agg = pd.DataFrame()

for i, chunk in enumerate(pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_A_CD', 'ALIGHT_A_CD', 'PAY_AMT', 'VACNTV_DIST']
)):
    chunk['RIDE_A_CD'] = chunk['RIDE_A_CD'].astype(str)
    chunk['ALIGHT_A_CD'] = chunk['ALIGHT_A_CD'].astype(str)
    
    grp = chunk.groupby(['RIDE_A_CD', 'ALIGHT_A_CD']).agg(
        trips=('PAY_AMT', 'count'),
        pay_sum=('PAY_AMT', 'sum'),
        vacntv_sum=('VACNTV_DIST', 'sum')
    ).reset_index()
    
    od_agg = pd.concat([od_agg, grp], ignore_index=True)
    
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} 처리 완료')
        mem_usage()
    del chunk, grp
    gc.collect()

od_final = od_agg.groupby(['RIDE_A_CD', 'ALIGHT_A_CD']).sum().reset_index()

del od_agg
gc.collect()

print(f'OD 쌍 수: {len(od_final):,}')
print(f'총 통행량: {od_final["trips"].sum():,}')
mem_usage()

## 2. 비대칭 지수 산출

비대칭 지수(Asymmetry Index) = `|V(A,B) - V(B,A)| / (V(A,B) + V(B,A))`

여기서 V(A,B)는 A에서 B로의 통행량이다. 이 지수는 0(완전 대칭)에서 1(완전 편도) 사이의 값을 가지며, 값이 클수록 한 방향 수요가 지배적임을 의미한다. 높은 비대칭 지수를 보이는 OD 쌍은 빈차 복귀 비용이 크므로 공유 모빌리티나 노선형 택시 도입의 우선 대상이 된다.


In [ ]:
# A->B와 B->A를 매칭
od_ab = od_final.rename(columns={
    'RIDE_A_CD': 'A', 'ALIGHT_A_CD': 'B',
    'trips': 'ab_trips', 'pay_sum': 'ab_pay', 'vacntv_sum': 'ab_vacntv'
})

od_ba = od_final.rename(columns={
    'RIDE_A_CD': 'B', 'ALIGHT_A_CD': 'A',
    'trips': 'ba_trips', 'pay_sum': 'ba_pay', 'vacntv_sum': 'ba_vacntv'
})

# 양방향 매칭
od_pair = od_ab.merge(od_ba, on=['A', 'B'], how='outer').fillna(0)

# 중복 제거 (A<B만 유지)
od_pair = od_pair[od_pair['A'] < od_pair['B']].copy()

# 비대칭 지수
od_pair['total'] = od_pair['ab_trips'] + od_pair['ba_trips']
od_pair['asymmetry'] = (
    abs(od_pair['ab_trips'] - od_pair['ba_trips']) / od_pair['total'].replace(0, np.nan)
).fillna(0)

# 의미 있는 통행량 필터 (최소 100건 이상)
od_sig = od_pair[od_pair['total'] >= 100].copy()

del od_ab, od_ba
gc.collect()

print(f'유효 OD쌍 수 (100건 이상): {len(od_sig):,}')
print(f'\n비대칭 지수 통계:')
print(od_sig['asymmetry'].describe())

## 3. 비대칭 지수 상위 20개 OD 쌍 바차트

상위 비대칭 OD 쌍을 시각화하여 서울시 내 편도 수요가 극단적인 구간을 파악한다. 이러한 구간은 주로 업무지구-주거지구 쌍(예: 강남-관악), 교통 허브-주거지구 쌍(예: 서울역-은평) 등에서 나타날 것으로 예상된다.


In [ ]:
top20_asym = od_sig.nlargest(20, 'asymmetry').copy()
top20_asym['label'] = top20_asym['A'] + ' <-> ' + top20_asym['B']
top20_asym = top20_asym.sort_values('asymmetry')

print('=== 비대칭 지수 상위 20개 OD쌍 ===')
print(top20_asym[['A', 'B', 'ab_trips', 'ba_trips', 'total', 'asymmetry']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

y = range(len(top20_asym))
bar_height = 0.35

ax.barh([yi + bar_height/2 for yi in y],
        top20_asym['ab_trips'].astype(int), height=bar_height,
        color='#1976d2', label='A -> B', edgecolor='white')
ax.barh([yi - bar_height/2 for yi in y],
        top20_asym['ba_trips'].astype(int), height=bar_height,
        color='#ef5350', label='B -> A', edgecolor='white')

ax.set_yticks(list(y))
ax.set_yticklabels(top20_asym['label'], fontsize=9)
ax.set_xlabel('통행량 (건)')
ax.set_title('비대칭 OD 상위 20쌍 - 양방향 통행량 비교')
ax.legend()

# 비대칭 지수 표시
for yi, (_, row) in zip(y, top20_asym.iterrows()):
    max_val = max(row['ab_trips'], row['ba_trips'])
    ax.text(max_val * 1.02, yi, f'{row["asymmetry"]:.2f}',
            va='center', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

## 4. 시간대별 OD 반전 패턴 탐지

출근 시간(7-9시)과 퇴근 시간(17-19시)에 OD 방향이 반전되는 쌍을 탐지한다. 이러한 반전 패턴은 도시의 직주 분리(jobs-housing imbalance) 구조를 직접적으로 반영하며, Liu et al. (2013)이 뉴욕시 택시 데이터에서 확인한 "tidal flow" 현상과 동일한 메커니즘이다.

출퇴근 OD 반전이 뚜렷한 쌍은 **시간대별 차등 요금** 또는 **역방향 할인** 정책의 대상 후보가 된다.


In [ ]:
# 시간대별 OD 집계 (출퇴근)
od_hour_agg = pd.DataFrame()

for i, chunk in enumerate(pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_DTIME', 'RIDE_A_CD', 'ALIGHT_A_CD']
)):
    chunk['RIDE_DTIME'] = chunk['RIDE_DTIME'].astype(str)
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype('int8')
    
    # 출근(7-9) 또는 퇴근(17-19)만 필터
    chunk['period'] = np.where(
        chunk['hour'].between(7, 9), 'morning',
        np.where(chunk['hour'].between(17, 19), 'evening', 'other')
    )
    chunk = chunk[chunk['period'] != 'other']
    
    if len(chunk) == 0:
        del chunk
        gc.collect()
        continue
    
    chunk['RIDE_A_CD'] = chunk['RIDE_A_CD'].astype(str)
    chunk['ALIGHT_A_CD'] = chunk['ALIGHT_A_CD'].astype(str)
    
    grp = chunk.groupby(['period', 'RIDE_A_CD', 'ALIGHT_A_CD']).size().reset_index(name='trips')
    od_hour_agg = pd.concat([od_hour_agg, grp], ignore_index=True)
    
    del chunk, grp
    gc.collect()

od_hour_final = od_hour_agg.groupby(['period', 'RIDE_A_CD', 'ALIGHT_A_CD'])['trips'].sum().reset_index()

del od_hour_agg
gc.collect()

print(f'출퇴근 OD 데이터 수: {len(od_hour_final):,}')
mem_usage()

In [ ]:
# 출근 A->B vs 퇴근 B->A 반전 패턴 탐지
morning = od_hour_final[od_hour_final['period'] == 'morning'].copy()
morning = morning.rename(columns={'RIDE_A_CD': 'A', 'ALIGHT_A_CD': 'B', 'trips': 'morning_ab'})
morning = morning[['A', 'B', 'morning_ab']]

evening = od_hour_final[od_hour_final['period'] == 'evening'].copy()
# 퇴근: B->A (반전)
evening_rev = evening.rename(columns={'RIDE_A_CD': 'B', 'ALIGHT_A_CD': 'A', 'trips': 'evening_ba'})
evening_rev = evening_rev[['A', 'B', 'evening_ba']]

# 출근 A->B와 퇴근 B->A 매칭
reversal = morning.merge(evening_rev, on=['A', 'B'], how='inner')

# 출근 방향이 아닌 방향도 가져오기 (morning B->A, evening A->B)
morning_ba = od_hour_final[od_hour_final['period'] == 'morning'].copy()
morning_ba = morning_ba.rename(columns={'RIDE_A_CD': 'B', 'ALIGHT_A_CD': 'A', 'trips': 'morning_ba'})
morning_ba = morning_ba[['A', 'B', 'morning_ba']]

evening_ab = od_hour_final[od_hour_final['period'] == 'evening'].copy()
evening_ab = evening_ab.rename(columns={'RIDE_A_CD': 'A', 'ALIGHT_A_CD': 'B', 'trips': 'evening_ab'})
evening_ab = evening_ab[['A', 'B', 'evening_ab']]

reversal = reversal.merge(morning_ba, on=['A', 'B'], how='left').fillna(0)
reversal = reversal.merge(evening_ab, on=['A', 'B'], how='left').fillna(0)

# 반전 지수: 출근에 A->B 우세 & 퇴근에 B->A 우세
reversal['morning_ratio'] = (
    reversal['morning_ab'] / (reversal['morning_ab'] + reversal['morning_ba']).replace(0, np.nan)
).fillna(0.5)
reversal['evening_ratio'] = (
    reversal['evening_ba'] / (reversal['evening_ab'] + reversal['evening_ba']).replace(0, np.nan)
).fillna(0.5)

# 반전 점수: 출근에 A->B 비율이 높고, 퇴근에 B->A 비율이 높을수록 큰 값
reversal['reversal_score'] = reversal['morning_ratio'] * reversal['evening_ratio']
reversal['total_trips'] = (
    reversal['morning_ab'] + reversal['morning_ba'] +
    reversal['evening_ab'] + reversal['evening_ba']
)

# 의미 있는 통행량 필터
reversal_sig = reversal[reversal['total_trips'] >= 50].copy()
reversal_sig = reversal_sig.sort_values('reversal_score', ascending=False)

del morning, evening, evening_rev, morning_ba, evening_ab
gc.collect()

print('=== 출퇴근 OD 반전 패턴 Top 20 ===')
print(reversal_sig.head(20)[[
    'A', 'B', 'morning_ab', 'morning_ba',
    'evening_ab', 'evening_ba', 'morning_ratio', 'evening_ratio', 'reversal_score'
]].to_string(index=False))

In [ ]:
# 반전 패턴 시각화: Top 15
top15_rev = reversal_sig.head(15).copy()
top15_rev['label'] = top15_rev['A'] + ' <-> ' + top15_rev['B']
top15_rev = top15_rev.sort_values('reversal_score')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

y = range(len(top15_rev))
h = 0.35

# 출근 시간
axes[0].barh([yi + h/2 for yi in y], top15_rev['morning_ab'].astype(int),
             height=h, color='#1976d2', label='A -> B')
axes[0].barh([yi - h/2 for yi in y], top15_rev['morning_ba'].astype(int),
             height=h, color='#ef5350', label='B -> A')
axes[0].set_yticks(list(y))
axes[0].set_yticklabels(top15_rev['label'], fontsize=8)
axes[0].set_title('출근 시간 (7-9시)')
axes[0].set_xlabel('통행량')
axes[0].legend()

# 퇴근 시간
axes[1].barh([yi + h/2 for yi in y], top15_rev['evening_ab'].astype(int),
             height=h, color='#1976d2', label='A -> B')
axes[1].barh([yi - h/2 for yi in y], top15_rev['evening_ba'].astype(int),
             height=h, color='#ef5350', label='B -> A')
axes[1].set_yticks(list(y))
axes[1].set_yticklabels(top15_rev['label'], fontsize=8)
axes[1].set_title('퇴근 시간 (17-19시)')
axes[1].set_xlabel('통행량')
axes[1].legend()

plt.suptitle('출퇴근 시간 OD 반전 패턴 Top 15', fontsize=14)
plt.tight_layout()
plt.show()

del top15_rev
gc.collect()

## 5. 빈차 낭비 추정

비대칭이 큰 OD에서 편도 수요 후 빈차로 복귀하는 택시의 운행거리를 추정한다. 이 "빈차 거리 낭비"는 연료비, 기회비용, 탄소 배출의 직접적 원인이다.

추정 방법: 비대칭 OD 쌍에서 `|V(A,B) - V(B,A)|` 만큼의 택시가 편도 빈차 복귀를 한다고 가정하고, OD 간 평균 운행거리를 곱하여 총 빈차거리를 산출한다. 이는 보수적 추정치(lower bound)에 해당한다.


In [ ]:
# 비대칭 상위 OD에서 빈차 낭비 추정
# 편도 수요 차이 = |A->B - B->A| = 빈차로 복귀해야 하는 건수
od_sig_copy = od_sig.copy()
od_sig_copy['excess_trips'] = abs(od_sig_copy['ab_trips'] - od_sig_copy['ba_trips']).astype(int)

# 평균 빈차거리 (총 빈차거리 / 총 통행)
od_sig_copy['avg_vacntv_ab'] = (
    od_sig_copy['ab_vacntv'] / od_sig_copy['ab_trips'].replace(0, np.nan)
).fillna(0)
od_sig_copy['avg_vacntv_ba'] = (
    od_sig_copy['ba_vacntv'] / od_sig_copy['ba_trips'].replace(0, np.nan)
).fillna(0)

# 편도 초과 방향의 빈차거리로 낭비 추정
od_sig_copy['wasted_vacntv'] = np.where(
    od_sig_copy['ab_trips'] > od_sig_copy['ba_trips'],
    od_sig_copy['excess_trips'] * od_sig_copy['avg_vacntv_ba'],  # B->A 방향 빈차
    od_sig_copy['excess_trips'] * od_sig_copy['avg_vacntv_ab']   # A->B 방향 빈차
)

total_wasted_km = od_sig_copy['wasted_vacntv'].sum() / 1000
total_excess = od_sig_copy['excess_trips'].sum()

# 상위 20개
top20_waste = od_sig_copy.nlargest(20, 'wasted_vacntv')

print(f'=== 비대칭 OD 빈차 낭비 추정 ===')
print(f'총 편도 초과 건수: {total_excess:,}')
print(f'추정 빈차 낭비 거리: {total_wasted_km:,.1f} km')
print(f'\n빈차 낭비 Top 20 OD:')
print(top20_waste[['A', 'B', 'ab_trips', 'ba_trips', 'excess_trips',
                    'wasted_vacntv', 'asymmetry']].to_string(index=False))

In [ ]:
top20_w = top20_waste.copy()
top20_w['label'] = top20_w['A'] + ' <-> ' + top20_w['B']
top20_w = top20_w.sort_values('wasted_vacntv')
top20_w['wasted_km'] = top20_w['wasted_vacntv'] / 1000

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top20_w['label'], top20_w['wasted_km'], color='#ff7043', edgecolor='white')
ax.set_xlabel('추정 빈차 낭비 (km)')
ax.set_title('비대칭 OD 빈차 낭비 Top 20')
plt.tight_layout()
plt.show()

del top20_w, top20_waste
gc.collect()

## 6. 대중교통 혼잡 회피 수요 추정

출근 시간 도심 방향 택시가 지하철 반대 방향보다 많은지를 분석한다. 이를 통해 택시 수요의 일부가 대중교통 혼잡을 회피하기 위한 **대체 수요(substitution demand)**인지를 검증한다.

만약 지하철 혼잡 노선과 겹치는 OD에서 택시 수요가 유의미하게 높다면, 이는 대중교통 용량 확충이 택시 수요에 영향을 미칠 수 있음을 시사한다.


In [ ]:
# 도심/강남 행정동 코드 매핑 (실데이터에 맞게 조정 필요)
# 일반적인 서울 행정동 코드 기준
# 도심권: 종로구(11010), 중구(11020), 용산구(11030)
# 강남권: 강남구(11230), 서초구(11220), 송파구(11240)

# 실제 데이터의 행정동 코드 분포 확인
area_codes = set()
for chunk in pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_A_CD']
):
    area_codes.update(chunk['RIDE_A_CD'].astype(str).unique())
    if len(area_codes) > 500:  # 충분히 수집되면 중단
        break
    del chunk
    gc.collect()

area_list = sorted(area_codes)
print(f'행정동 코드 수: {len(area_list)}')
print(f'코드 샘플: {area_list[:20]}')
print(f'코드 길이 분포: {set(len(c) for c in area_list)}')

In [ ]:
# WARNING: 아래 행정동 코드(11010 등)는 데모용 가정값입니다.
# 실제 데이터의 RIDE_A_CD/ALIGHT_A_CD 코드 체계를 확인 후 수정하세요.
# 행정동 코드 앞 5자리로 구 단위 매핑
# 도심: 종로(11010), 중구(11020), 용산(11030)
# 강남: 강남(11230), 서초(11220)
# 여의도: 영등포(11150)

DOWNTOWN = ['11010', '11020', '11030']   # 도심권
GANGNAM = ['11230', '11220']              # 강남권

# 출근시간(7-9시) 도심->강남 vs 강남->도심
commute_agg = {'downtown_to_gangnam': 0, 'gangnam_to_downtown': 0,
               'downtown_to_other': 0, 'gangnam_to_other': 0}

for chunk in pd.read_csv(
    D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE_D012,
    usecols=['RIDE_DTIME', 'RIDE_A_CD', 'ALIGHT_A_CD']
):
    chunk['RIDE_DTIME'] = chunk['RIDE_DTIME'].astype(str)
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype('int8')
    chunk = chunk[chunk['hour'].between(7, 9)]  # 출근 시간만
    
    if len(chunk) == 0:
        del chunk
        gc.collect()
        continue
    
    chunk['ride_gu'] = chunk['RIDE_A_CD'].astype(str).str[:5]
    chunk['alight_gu'] = chunk['ALIGHT_A_CD'].astype(str).str[:5]
    
    # 도심 -> 강남
    mask_dg = chunk['ride_gu'].isin(DOWNTOWN) & chunk['alight_gu'].isin(GANGNAM)
    commute_agg['downtown_to_gangnam'] += mask_dg.sum()
    
    # 강남 -> 도심
    mask_gd = chunk['ride_gu'].isin(GANGNAM) & chunk['alight_gu'].isin(DOWNTOWN)
    commute_agg['gangnam_to_downtown'] += mask_gd.sum()
    
    # 도심 -> 기타
    mask_do = chunk['ride_gu'].isin(DOWNTOWN) & ~chunk['alight_gu'].isin(GANGNAM)
    commute_agg['downtown_to_other'] += mask_do.sum()
    
    # 강남 -> 기타
    mask_go = chunk['ride_gu'].isin(GANGNAM) & ~chunk['alight_gu'].isin(DOWNTOWN)
    commute_agg['gangnam_to_other'] += mask_go.sum()
    
    del chunk
    gc.collect()

print('=== 출근시간(7-9시) 도심/강남 간 택시 수요 ===')
for k, v in commute_agg.items():
    print(f'  {k}: {v:,} 건')

# 지하철 반대 방향 비교
# 지하철 주 혼잡: 강남->도심 (2호선 등)
# 택시 혼잡 회피: 도심->강남이 강남->도심보다 많으면 역방향 수요
if commute_agg['gangnam_to_downtown'] > 0:
    ratio = commute_agg['downtown_to_gangnam'] / commute_agg['gangnam_to_downtown']
    print(f'\n도심->강남 / 강남->도심 비율: {ratio:.2f}')
    if ratio > 1:
        print('-> 출근시간 도심->강남 택시가 더 많음 (지하철 혼잡 회피 수요 존재 가능)')
    else:
        print('-> 출근시간 강남->도심 택시가 더 많음 (일반적 출근 패턴)')

mem_usage()

In [ ]:
# 시각화: 출근시간 도심-강남 흐름
labels = ['도심 -> 강남', '강남 -> 도심', '도심 -> 기타', '강남 -> 기타']
values = [commute_agg['downtown_to_gangnam'], commute_agg['gangnam_to_downtown'],
          commute_agg['downtown_to_other'], commute_agg['gangnam_to_other']]
colors = ['#1976d2', '#ef5350', '#90caf9', '#ef9a9a']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white')
ax.set_title('출근시간(7-9시) 도심/강남 간 택시 통행량')
ax.set_ylabel('통행량 (건)')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 7. 요약

비대칭 OD 분석 결과를 종합한다. 핵심 인사이트는 (1) 비대칭 지수가 높은 OD 쌍의 지리적 패턴, (2) 출퇴근 OD 반전의 규모, (3) 빈차 낭비의 경제적 비용이다. 이 결과는 노선형 택시(route-based taxi) 도입, 역방향 할인 요금제, 공유 모빌리티 서비스 기획의 데이터 기반 근거로 활용할 수 있다.


In [ ]:
print('=' * 70)
print('비대칭 OD 분석 요약')
print('=' * 70)

# 비대칭 OD Top 10
top10 = od_sig.nlargest(10, 'asymmetry')
print('\n[1] 가장 비대칭적인 OD Top 10')
for _, row in top10.iterrows():
    dominant = f'{row["A"]}->{row["B"]}' if row['ab_trips'] > row['ba_trips'] else f'{row["B"]}->{row["A"]}'
    print(f'    {row["A"]} <-> {row["B"]}: '
          f'비대칭={row["asymmetry"]:.2f}, '
          f'A->B={int(row["ab_trips"]):,}, B->A={int(row["ba_trips"]):,}, '
          f'우세방향={dominant}')

# 반전 패턴
print('\n[2] 출퇴근 시간 반전 패턴 Top 5')
for _, row in reversal_sig.head(5).iterrows():
    print(f'    {row["A"]} <-> {row["B"]}: '
          f'출근 A->B={int(row["morning_ab"]):,} vs B->A={int(row["morning_ba"]):,} | '
          f'퇴근 A->B={int(row["evening_ab"]):,} vs B->A={int(row["evening_ba"]):,}')

# 빈차 낭비
print(f'\n[3] 빈차 낭비 규모')
print(f'    총 편도 초과 건수: {total_excess:,}')
print(f'    추정 빈차 낭비 거리: {total_wasted_km:,.1f} km')

# 대중교통 회피
print(f'\n[4] 출근시간 도심-강남 택시 수요')
print(f'    도심->강남: {commute_agg["downtown_to_gangnam"]:,} 건')
print(f'    강남->도심: {commute_agg["gangnam_to_downtown"]:,} 건')

print('=' * 70)
mem_usage()

## 8. [보강] 시계열 추세

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(D012_PATH if 'D012_PATH' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")

---

## References

1. Liu, L., Andris, C., & Ratti, C. (2013). Revealing travel patterns and city structure with taxi trip data. *Journal of Transport Geography*, 43, 78-90.
2. Cramer, J., & Krueger, A. B. (2016). Disruptive Change in the Taxi Business: The Case of Uber. *American Economic Review*, 106(5), 177-182.
3. Zheng, Z., Rasouli, S., & Timmermans, H. (2022). Analyses of taxi trip patterns in New York City. *Transportation Research Part C*, 138, 103616.
